<details>
<summary><b>Info</b></summary>

**Last Execution:** 2026-07-25

| Package | Version |
|---------|---------|
| **nnsight** | **0.8** |
| Python | 3.12.3 |
| torch | 2.10.0+cu128 |
| transformers | 5.2.0 |

</details>


# Scan

`model.scan()` runs the forward pass under PyTorch's `FakeTensorMode`: tensors carry real shapes and dtypes, but no data, no kernels run, and — crucially — **the model is never dispatched**. That means you can inspect activation shapes or debug interventions on an undispatched (meta) model, before any weights are loaded into memory.

## Setup

We construct a `TransformersModel`. In nnsight 0.8 the model is built lazily on the meta device — its architecture is known, but no real weights are loaded until the first real forward. `model.dispatched` tells us whether weights have been loaded yet.

In [1]:
import nnsight
import torch
from nnsight.modeling.transformers import TransformersModel

model = TransformersModel("openai-community/gpt2", device_map="auto")

print(f"Dispatched (weights loaded)? {model.dispatched}")

/home/localjadenfk/miniconda3/envs/ndif2/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Dispatched (weights loaded)? False


## Getting Shape Information

Use `model.scan()` like `model.trace()`, but no real computation happens. You can inspect `.shape` on any module's output to learn its dimensions. Because scan never dispatches, this runs without loading GPT-2's weights.

In [2]:
with model.scan("The Eiffel Tower is in the city of"):
    hidden_dim = nnsight.save(model.transformer.h[0].output.shape[-1])
    seq_len = nnsight.save(model.transformer.h[0].output.shape[1])
    vocab_size = nnsight.save(model.lm_head.output.shape[-1])

print(f"Hidden dim: {hidden_dim}")
print(f"Sequence length: {seq_len}")
print(f"Vocab size: {vocab_size}")

print(f"\nStill undispatched after scan? {not model.dispatched}")

Hidden dim: 768
Sequence length: 10
Vocab size: 50257

Still undispatched after scan? True


<details class="admonition warning">
<summary>You must save values to access them outside scan</summary>

`model.scan()` is a tracing context just like `model.trace()`. Values defined inside it are only valid within the block. Use `nnsight.save()` for non-tensor values like shape integers, or `.save()` for tensors. Note that saved tensors are `FakeTensor`s — you can read their `.shape`/`.dtype`, but they hold no data.

</details>

## Values Inside Scan Are Fake Tensors

Inside a scan, module outputs are `FakeTensor`s. They know their shape and dtype but carry no real data — so read metadata, not values.

In [3]:
with model.scan("The Eiffel Tower is in the city of"):
    hs = model.transformer.h[-1].output.save()

print(f"Type:  {type(hs).__name__}")
print(f"Shape: {tuple(hs.shape)}")
print(f"Dtype: {hs.dtype}")

Type:  FakeTensor
Shape: (1, 10, 768)
Dtype: torch.float32


## Inspecting Shapes with Print

You can `print()` inside a scan context to inspect shapes interactively — useful for exploring an unfamiliar model. Access modules in forward-pass order within an invoke, just like in `model.trace()`.

In [4]:
with model.scan("The Eiffel Tower is in the city of"):
    print(f"Embedding output: {model.transformer.wte.output.shape}")
    print(f"Layer 0 output:   {model.transformer.h[0].output.shape}")
    print(f"Layer 11 output:  {model.transformer.h[11].output.shape}")
    print(f"LM head output:   {model.lm_head.output.shape}")

Embedding output: torch.Size([1, 10, 768])
Layer 0 output:   torch.Size([1, 10, 768])
Layer 11 output:  torch.Size([1, 10, 768])


LM head output:   torch.Size([1, 10, 50257])


## Debugging Interventions

Operations inside a scan raise the same shape errors a real forward would — so scan catches a broken intervention before you run the real model. This is especially useful for large models, where a failed run wastes significant time, and here it happens without ever loading the weights.

In [5]:
input_text = "The Eiffel Tower is in the city of"

# Bug: GPT-2's hidden dimension is 768, but this steering vector is size 1024
wrong_vector = torch.randn(1024)

try:
    with model.scan(input_text):
        model.transformer.h[5].output[:, -1, :] += wrong_vector
except RuntimeError as e:
    print(f"Caught error: {e}")

E0725 18:34:00.929000 1751221 site-packages/torch/_subclasses/fake_tensor.py:3090] failed while attempting to run meta for aten.add_.Tensor
E0725 18:34:00.929000 1751221 site-packages/torch/_subclasses/fake_tensor.py:3090] Traceback (most recent call last):
E0725 18:34:00.929000 1751221 site-packages/torch/_subclasses/fake_tensor.py:3090]   File "/home/localjadenfk/miniconda3/envs/ndif2/lib/python3.12/site-packages/torch/_subclasses/fake_tensor.py", line 3086, in _dispatch_impl
E0725 18:34:00.929000 1751221 site-packages/torch/_subclasses/fake_tensor.py:3090]     r = func(*args, **kwargs)
E0725 18:34:00.929000 1751221 site-packages/torch/_subclasses/fake_tensor.py:3090]         ^^^^^^^^^^^^^^^^^^^^^
E0725 18:34:00.929000 1751221 site-packages/torch/_subclasses/fake_tensor.py:3090]   File "/home/localjadenfk/miniconda3/envs/ndif2/lib/python3.12/site-packages/torch/_ops.py", line 875, in __call__
E0725 18:34:00.929000 1751221 site-packages/torch/_subclasses/fake_tensor.py:3090]     retur

Caught error: Attempting to broadcast a dimension of length 1024 at -1! Mismatching argument at index 1 had torch.Size([1024]); but expected shape should be broadcastable to [1, 768]


In [6]:
# Fixed version — a correctly-sized vector passes the shape check
right_vector = torch.randn(768)

with model.scan(input_text):
    model.transformer.h[5].output[:, -1, :] += right_vector
    print("Intervention shape check passed!")

Intervention shape check passed!


## Using Scan for Dynamic Dimensions

Scan is useful when you need to know a model's dimensions to construct tensors for interventions — like steering vectors or noise. Here we read the hidden dimension under fake tensors (no weights loaded), then run a real trace, which dispatches the model on demand.

In [7]:
with model.scan("test"):
    dim = nnsight.save(model.transformer.h[0].output.shape[-1])

print(f"Hidden dimension: {dim}")
print(f"Dispatched after scan? {model.dispatched}")

# Now use the dimension in a real trace
steering_vector = torch.randn(dim)

with model.trace("The Eiffel Tower is in the city of"):
    device = model.transformer.h[5].output.device
    model.transformer.h[5].output[:, -1, :] += steering_vector.to(device)
    logits = model.lm_head.output.save()

print(f"Steered prediction: {model.tokenizer.decode(logits[0, -1].argmax(dim=-1))}")
print(f"Dispatched after trace? {model.dispatched}")

Hidden dimension: 768
Dispatched after scan? False


Steered prediction:  Paris
Dispatched after trace? True


<details class="admonition tip">
<summary>When to use scan</summary>

- **Exploring unfamiliar models** — quickly check shapes at every layer without loading weights or running the full model
- **Debugging interventions** — catch shape mismatches and index errors before a costly real run
- **Dynamic tensor construction** — learn hidden dimensions to build correctly-sized steering vectors, probes, or adapters
- **Remote execution** — validate interventions locally before sending them to NDIF

</details>

<details class="admonition note">
<summary>Gotchas</summary>

- Outputs are `FakeTensor`s — read `.shape`/`.dtype`/`.device`, not data.
- `.save()` is required just like in `model.trace()`. Use `nnsight.save()` for non-tensor values (ints, shapes, lists).
- Access modules in forward-pass order within an invoke (same rule as trace).
- Some ops lack a fake/meta kernel and may raise inside scan even if they work in a real forward. Move that code out of scan if you hit this.

</details>